In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [3]:
dim_employee = pd.read_excel('dim_employee.xlsx')
fact_workload = pd.read_excel('fact_workload.xlsx')
fact_record = pd.read_excel('fact_record.xlsx')

# Primary Data Analysis

### 1 Analyze salary trends over tenure and department

In [4]:
merge_df = pd.merge(dim_employee,fact_record, on = "Employee_ID" , how = 'inner')
avg_dept_salary_trends = merge_df.groupby('Department')[['Years_At_Company','Monthly_Salary']].mean().sort_values('Monthly_Salary',ascending = False).round(2)
desired_output=pd.concat([avg_dept_salary_trends.head(3),avg_dept_salary_trends.tail(3)])
desired_output

,Years_At_Company,Monthly_Salary
Department,,
Legal,5.99,144473.37
Operations,5.80,141030.16
Customer Support,5.86,140720.25
IT,5.91,139781.87
Engineering,5.76,139174.97
Finance,5.73,138635.16


### 2 Identify most common education levels per role

In [5]:
most_common_edu = dim_employee.groupby('Job_Title')['Education_Level'].agg(pd.Series.mode)
most_common_edu

Job_Title
Analyst           Bachelor
Consultant        Bachelor
Developer         Bachelor
Director            Master
Engineer          Bachelor
Manager           Bachelor
Specialist        Bachelor
Sr. Manager         Master
Technician     High School
Name: Education_Level, dtype: object

### 3 Determine the impact of remote work on performance

In [6]:
merge_df_1 = pd.merge(fact_workload,fact_record, on = "Employee_ID" , how = 'inner')
remote_work_performance = merge_df_1.groupby('Remote_Work_Frequency')[['Employee_Satisfaction_Score','Performance_Score']].mean().round(2)
remote_work_performance

,Employee_Satisfaction_Score,Performance_Score
Remote_Work_Frequency,,
0,3.49,3.20
20,3.49,3.19
50,3.48,3.18
80,3.46,3.19
100,3.48,3.21


### 4 Analyze employee retention trends

In [7]:
merge_df = pd.merge(dim_employee,fact_record, on = "Employee_ID" , how = 'inner')
merge_df["Age_groups"] = pd.cut(x=merge_df["Age"],bins=[18,30,45,60],labels=["18-30", "31-45", "45-60"],right=True)
employee_retention_by_age =  merge_df.pivot_table(index = 'Age_groups', columns = 'Resigned', values='Employee_ID',aggfunc='count').fillna(0)
employee_retention_by_age['Percent_Resign'] = (employee_retention_by_age[True]*100)/(employee_retention_by_age[True]+employee_retention_by_age[False])
employee_retention_by_age

C:\Users\adity\AppData\Local\Temp\ipykernel_1660\2658106181.py:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  employee_retention_by_age =  merge_df.pivot_table(index = 'Age_groups', columns = 'Resigned', values='Employee_ID',aggfunc='count').fillna(0)


Resigned,False,True,Percent_Resign
Age_groups,,,
18-30,16473,12159,42.466471
31-45,25369,10223,28.722747
45-60,28726,4700,14.060911


### 5 Department-wise education level distribution

In [8]:
dept_edu_dist = dim_employee.pivot_table(index = 'Department', columns = 'Education_Level',values='Employee_ID',aggfunc='count').fillna(0)
dept_edu_dist

Education_Level,Bachelor,High School,Master,PhD
Department,,,,
Customer Support,8318,1593,4390,473
Engineering,11295,2117,5927,670
Finance,2795,534,1506,170
HR,2247,459,1161,160
IT,11261,2142,5841,652
Legal,1110,222,600,77
Marketing,2313,438,1120,114
Operations,8363,1616,4506,501
Sales,8723,1538,4566,482


### 6 Year-wise hiring trends in the company

In [9]:
fact_record['Hire_Date'] = pd.to_datetime(fact_record['Hire_Date'])
fact_record.dtypes

index                         int64
Employee_ID                   int64
Hire_Date            datetime64[ns]
Years_At_Company              int64
Performance_Score             int64
Monthly_Salary                int64
Promotions                    int64
Resigned                       bool
dtype: object

In [10]:
yearly_hiring = fact_record[['Employee_ID','Hire_Date']]
yearly_hiring['Year'] = yearly_hiring['Hire_Date'].dt.year
yearly_hiring_merge = pd.merge(dim_employee,yearly_hiring, on= 'Employee_ID')
yearly_hiring_pivot = yearly_hiring_merge.pivot_table(index = 'Year', columns = 'Department',values='Employee_ID',aggfunc='count').fillna(0)
yearly_hiring_pivot.round(0)

C:\Users\adity\AppData\Local\Temp\ipykernel_1660\3951655090.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  yearly_hiring['Year'] = yearly_hiring['Hire_Date'].dt.year


Department,Customer Support,Engineering,Finance,HR,IT,Legal,Marketing,Operations,Sales
Year,,,,,,,,,
1989,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1990,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1991,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1992,0.0,3.0,0.0,0.0,4.0,0.0,0.0,0.0,1.0
1993,2.0,2.0,1.0,0.0,2.0,0.0,0.0,1.0,1.0
1994,1.0,1.0,0.0,1.0,6.0,0.0,1.0,1.0,3.0
1995,2.0,7.0,1.0,1.0,7.0,1.0,2.0,4.0,7.0
1996,6.0,6.0,2.0,3.0,14.0,1.0,3.0,4.0,8.0
1997,11.0,12.0,4.0,3.0,14.0,0.0,8.0,13.0,14.0


### 7 Gender distribution across departments

In [11]:
gender_dist = dim_employee.pivot_table(index = 'Department', columns = 'Gender',values='Employee_ID',aggfunc='count',margins = True).fillna(0)
gender_dist

Gender,Female,Male,All
Department,,,
Customer Support,6574,8200,14774
Engineering,9099,10910,20009
Finance,2291,2714,5005
HR,1842,2185,4027
IT,8954,10942,19896
Legal,870,1139,2009
Marketing,1808,2177,3985
Operations,6710,8276,14986
Sales,6852,8457,15309


### 8 Department, Job title attrition rates

In [12]:
merge_df = pd.merge(dim_employee,fact_record, on = "Employee_ID" , how = 'inner')
employee_retention_by_dept =  merge_df.pivot_table(index = 'Department', columns = 'Resigned', values='Employee_ID',aggfunc='count').fillna(0)
employee_retention_by_dept['Percent_Resign'] = (employee_retention_by_dept[True]*100)/(employee_retention_by_dept[True]+employee_retention_by_dept[False])
employee_retention_by_dept

Resigned,False,True,Percent_Resign
Department,,,
Customer Support,8659,6115,41.390280
Engineering,15389,4620,23.089610
Finance,4335,670,13.386613
HR,3470,557,13.831636
IT,15340,4556,22.899075
Legal,1757,252,12.543554
Marketing,2306,1679,42.132999
Operations,11585,3401,22.694515
Sales,8942,6367,41.589914


### 9 Job title and working hours correlation

In [13]:
merge_df_2 = pd.merge(dim_employee,fact_workload, on = "Employee_ID" , how = 'inner')
job_wise_work_hours = merge_df_2.groupby('Job_Title')['Work_Hours_Per_Week'].mean()
job_wise_work_hours

Job_Title
Analyst        45.213172
Consultant     45.111025
Developer      45.147010
Director       45.215292
Engineer       45.171361
Manager        45.173935
Specialist     45.158172
Sr. Manager    45.171092
Technician     45.153683
Name: Work_Hours_Per_Week, dtype: float64

In [14]:
merge_df_1 = pd.merge(fact_workload,fact_record, on = "Employee_ID" , how = 'inner')
merge_df_full = pd.merge(merge_df_1,dim_employee, on = "Employee_ID" , how = 'inner')
remote_work_perf_score = merge_df_full.pivot_table(index = 'Remote_Work_Frequency', columns = 'Department', values = 'Performance_Score',aggfunc = 'mean').fillna(0)
remote_work_perf_score.round(3)

Department,Customer Support,Engineering,Finance,HR,IT,Legal,Marketing,Operations,Sales
Remote_Work_Frequency,,,,,,,,,
0,3.202,3.213,3.198,3.193,3.191,3.243,3.222,3.207,3.193
20,3.190,3.190,3.223,3.222,3.193,3.219,3.186,3.186,3.174
50,3.166,3.189,3.212,3.204,3.158,3.162,3.191,3.201,3.187
80,3.157,3.180,3.236,3.192,3.205,3.167,3.137,3.211,3.213
100,3.206,3.228,3.180,3.315,3.187,3.126,3.225,3.229,3.219
